In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tkinter as tk

from tkinter import messagebox, ttk

In [3]:
df = pd.read_csv('mldataset_v1.csv')

In [4]:
# df.info()

In [5]:
df.columns.value_counts()

to        1
based     1
on        1
of        1
their     1
         ..
style     1
call      1
caller    1
phone     1
label     1
Name: count, Length: 334, dtype: int64

In [6]:
df.label.value_counts()

label
1    100
2    100
3    100
4    100
5    100
6    100
Name: count, dtype: int64

In [7]:
X = df.drop('label', axis=1)
y = df['label']
    

In [8]:
def preprocess_data(df, test_size=0.2, random_state=42):
    n_samples = len(X)
    n_test = int(n_samples * test_size)
    
    np.random.seed(random_state)
    indices = np.random.permutation(n_samples)
    
    test_indices = indices[:n_test]
    train_indices = indices[n_test:]
    
    X_train = X.iloc[train_indices]
    X_test = X.iloc[test_indices]
    y_train = y.iloc[train_indices]
    y_test = y.iloc[test_indices]
    
    return X_train, X_test, y_train, y_test

In [9]:
X_train, X_test, y_train, y_test = preprocess_data(df, test_size=0.2)

In [10]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(480, 333)
(480,)
(120, 333)
(120,)


In [11]:
class NaiveBayesMultinomial:
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self._classes = np.unique(y)
        n_classes = len(self._classes)

        self._priors = np.zeros(n_classes)
        self._feature_probs = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self._classes):
            X_c = X[y == c]
            self._priors[idx] = X_c.shape[0] / float(n_samples)

            total_count_per_word = X_c.sum(axis=0)
            total_word_in_class = X_c.sum()

            self._feature_probs[idx, :] = (total_count_per_word + 1) / (total_word_in_class + n_features)

            self.vocabulary = df.drop(columns='label').columns.tolist()

    def predict(self, X):
        y_pred = [self._predict_one(x) for x in X]
        return np.array(y_pred)


    def predict_single(self, text):
        if not isinstance(text, str): return ""

        text_clean = text.lower()
        text_clean = str.replace(r'[^a-z\s]', '', text_clean)
        text_user = text_clean.strip()

        vector_input = np.array([1 if kata in text_user else 0 for kata in self.vocabulary])

        return self._predict_one(vector_input)

    def _predict_one(self, x):
        posteriors = []

        for idx, c in enumerate(self._classes):
            prior = np.log(self._priors[idx])
            likelihood = np.sum(x * np.log(self._feature_probs[idx]))
            posterior = prior + likelihood
            posteriors.append(posterior)

        return self._classes[np.argmax(posteriors)]


In [12]:
model_nb = NaiveBayesMultinomial()
model_nb.fit(X_train.values, y_train.values)

In [13]:
akurasi_train = model_nb.predict(X_train.values)
train_accuracy = np.sum(akurasi_train == y_train) / len(X_train)
print(f"Train Accuracy: {train_accuracy:.4f}")

Train Accuracy: 0.9896


In [14]:
akurasi_test = model_nb.predict(X_test.values)
test_accuracy = np.sum(akurasi_test == y_test) / len(X_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.9333


In [15]:
cm_df_test = pd.crosstab(y_test, akurasi_test, rownames=['Aktual'], colnames=['Prediksi'])
cm_df_train = pd.crosstab(y_train, akurasi_train, rownames=['Aktual'], colnames=['Prediksi'])

print("Train")
for label2 in cm_df_train.index:
    TP2 = cm_df_train.loc[label2, label2]

    Total_Asli2 = cm_df_train.loc[label2].sum()
    FN2 = Total_Asli2 - TP2

    Total_Prediksi2 = cm_df_train[label2].sum()
    FP2 = Total_Prediksi2 - TP2

    precision2 = TP2 / (TP2 + FP2) if (TP2 + FP2) > 0 else 0
    recall2 = TP2 / (TP2 + FN2) if (TP2 + FN2) > 0 else 0
    f12 = (precision2 * recall2) / (precision2 + recall2) if (precision2 + recall2) > 0 else 0
    print(f"Kelas {label2} -> Precision: {precision2:.2f}, Recall: {recall2:.2f}, F1: {f12:.2f}")
    
print('\nTest')
for label in cm_df_test.index:
    TP = cm_df_test.loc[label, label]

    Total_Asli = cm_df_test.loc[label].sum()
    FN = Total_Asli - TP

    Total_Prediksi = cm_df_test[label].sum()
    FP = Total_Prediksi - TP

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    print(f"Kelas {label} -> Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")

Train
Kelas 1 -> Precision: 1.00, Recall: 0.94, F1: 0.48
Kelas 2 -> Precision: 1.00, Recall: 1.00, F1: 0.50
Kelas 3 -> Precision: 1.00, Recall: 1.00, F1: 0.50
Kelas 4 -> Precision: 0.98, Recall: 1.00, F1: 0.49
Kelas 5 -> Precision: 0.99, Recall: 1.00, F1: 0.50
Kelas 6 -> Precision: 0.97, Recall: 1.00, F1: 0.49

Test
Kelas 1 -> Precision: 1.00, Recall: 0.70, F1: 0.41
Kelas 2 -> Precision: 0.95, Recall: 1.00, F1: 0.49
Kelas 3 -> Precision: 0.95, Recall: 1.00, F1: 0.49
Kelas 4 -> Precision: 0.89, Recall: 0.94, F1: 0.46
Kelas 5 -> Precision: 0.94, Recall: 1.00, F1: 0.49
Kelas 6 -> Precision: 0.89, Recall: 1.00, F1: 0.47


In [16]:
def cek_data_baru():
    data_baru = entry1.get().strip()

    if not data_baru:
        messagebox.showwarning('Input Kosong', 'Input Yang Bener Kocak')
        return

    try:
        hasil = model_nb.predict_single(data_baru)
        messagebox.showinfo("Hasil Prediksi", f"Kategori: {hasil}")
        
    except Exception as e:
        messagebox.showerror("Error", f"Gagal prediksi: {e}")

root = tk.Tk()
root.configure(bg="#1E1D1F")

# frame
frame = tk.Frame(root, bg="#1E1D1F")
frame.configure(bg="#1E1D1F")
frame.place(relx=0.5,
            rely=0.40,
            anchor='center')
# label1
label1 = tk.Label(frame,text="Prediksi Teks",
                 font=("Arial", 48, "bold"),
                 bg="#1E1D1F",
                 fg="#FFFFFF")
label1.pack()

# entry1
entry1 = tk.Entry(frame,
                      font=("Arial", 25),
                      bg="gray",
                      fg="#FFFFFF")
entry1.pack(pady=20)

# button1
button1 = tk.Button(frame,
                       text="Prediksi!",
                       font=("Arial", 20, 'bold'),
                       command=cek_data_baru)
button1.pack(pady=10)

# label
label_hasil = tk.Label(frame,
                       text="...",
                       bg="#1E1D1F",
                       fg="#FFFFFF")
label_hasil.pack()

root.mainloop()

# ekhem


In [17]:
# # Cell 5: Implementasi Naive Bayes Multinomial Manual
# class NaiveBayesMultinomial:
#     """
#     Implementasi manual Naive Bayes Multinomial
#     untuk klasifikasi biner/multikelas.
#     """
    
#     def __init__(self, alpha=1.0):
#         """
#         Parameters:
#         -----------
#         alpha : float
#             Parameter smoothing (Laplace smoothing)
#         """
#         self.alpha = alpha
#         self.classes = None
#         self.class_priors = None
#         self.feature_probs = None
#         self.n_features = None
        
#     def fit(self, X, y):
#         """
#         Melatih model Naive Bayes Multinomial.
        
#         Parameters:
#         -----------
#         X : array-like, shape (n_samples, n_features)
#             Data fitur
#         y : array-like, shape (n_samples,)
#             Label kelas
#         """
#         # Identifikasi kelas unik
#         self.classes = np.unique(y)
#         n_classes = len(self.classes)
#         n_samples, self.n_features = X.shape
        
#         # Hitung prior probabilities P(y)
#         self.class_priors = np.zeros(n_classes)
#         for i, c in enumerate(self.classes):
#             self.class_priors[i] = np.sum(y == c) / n_samples
        
#         # Hitung conditional probabilities P(x_i|y)
#         # Untuk multinomial, kita hitung probabilitas setiap fitur untuk setiap kelas
#         self.feature_probs = np.zeros((n_classes, self.n_features))
        
#         for i, c in enumerate(self.classes):
#             # Ambil data untuk kelas c
#             X_c = X[y == c]
            
#             # Hitung jumlah kemunculan setiap fitur di kelas c
#             # Untuk multinomial, kita jumlahkan semua nilai fitur
#             feature_counts = np.sum(X_c, axis=0)
            
#             # Hitung total semua fitur di kelas c (untuk normalisasi)
#             total_counts = np.sum(feature_counts)
            
#             # Hitung probabilitas dengan Laplace smoothing
#             # P(x_i|y=c) = (count(x_i, c) + alpha) / (total_counts(c) + alpha * n_features)
#             self.feature_probs[i] = (feature_counts + self.alpha) / (total_counts + self.alpha * self.n_features)
        
#         # Untuk stabilitas numerik, ubah ke log
#         self.class_priors_log = np.log(self.class_priors)
#         self.feature_probs_log = np.log(self.feature_probs)
        
#         print(f"Model berhasil dilatih dengan {n_classes} kelas")
        
#     def predict(self, X):
#         """
#         Memprediksi kelas untuk data X.
        
#         Parameters:
#         -----------
#         X : array-like, shape (n_samples, n_features)
#             Data yang akan diprediksi
            
#         Returns:
#         --------
#         predictions : array, shape (n_samples,)
#             Prediksi kelas
#         """
#         n_samples = X.shape[0]
#         log_probs = np.zeros((n_samples, len(self.classes)))
        
#         # Hitung log posterior untuk setiap kelas
#         for i in range(len(self.classes)):
#             # Log prior untuk kelas i
#             log_probs[:, i] = self.class_priors_log[i]
            
#             # Tambahkan log likelihood untuk setiap fitur
#             # Untuk multinomial: log P(x|y) = Σ_j x_j * log(P(x_j|y))
#             for j in range(n_samples):
#                 # x_j * log(P(x_j|y)) untuk semua fitur
#                 log_probs[j, i] += np.sum(X[j] * self.feature_probs_log[i])
        
#         # Prediksi kelas dengan probabilitas tertinggi
#         predictions = self.classes[np.argmax(log_probs, axis=1)]
        
#         return predictions
    
#     def predict_proba(self, X):
#         """
#         Mengembalikan probabilitas untuk setiap kelas.
        
#         Parameters:
#         -----------
#         X : array-like, shape (n_samples, n_features)
#             Data yang akan diprediksi
            
#         Returns:
#         --------
#         probabilities : array, shape (n_samples, n_classes)
#             Probabilitas untuk setiap kelas
#         """
#         n_samples = X.shape[0]
#         log_probs = np.zeros((n_samples, len(self.classes)))
        
#         # Hitung log posterior untuk setiap kelas
#         for i in range(len(self.classes)):
#             # Log prior untuk kelas i
#             log_probs[:, i] = self.class_priors_log[i]
            
#             # Tambahkan log likelihood untuk setiap fitur
#             for j in range(n_samples):
#                 log_probs[j, i] += np.sum(X[j] * self.feature_probs_log[i])
        
#         # Konversi log probabilitas ke probabilitas normal
#         # Gunakan log-sum-exp trick untuk menghindari underflow
#         max_log_probs = np.max(log_probs, axis=1, keepdims=True)
#         exp_log_probs = np.exp(log_probs - max_log_probs)
#         probabilities = exp_log_probs / np.sum(exp_log_probs, axis=1, keepdims=True)
        
#         return probabilities
    
#     def score(self, X, y):
#         """
#         Menghitung akurasi pada data X dengan label y.
        
#         Parameters:
#         -----------
#         X : array-like, shape (n_samples, n_features)
#             Data fitur
#         y : array-like, shape (n_samples,)
#             Label sebenarnya
            
#         Returns:
#         --------
#         accuracy : float
#             Akurasi prediksi
#         """
#         predictions = self.predict(X)
#         accuracy = np.mean(predictions == y)
#         return accuracy

In [18]:
# # Cell 6: Training dan Evaluasi Model
# # Inisialisasi model
# model = NaiveBayesMultinomial(alpha=1.0)

# # Latih model
# print("=== TRAINING MODEL ===")
# model.fit(X_train, y_train)

# # Prediksi pada data latih
# y_train_pred = model.predict(X_train)
# train_accuracy = np.mean(y_train_pred == y_train)
# print(f"\nAkurasi data latih: {train_accuracy:.4f}")

# # Prediksi pada data uji
# y_test_pred = model.predict(X_test)
# test_accuracy = np.mean(y_test_pred == y_test)
# print(f"Akurasi data uji: {test_accuracy:.4f}")

# # Hitung confusion matrix manual
# def compute_confusion_matrix(y_true, y_pred):
#     """Menghitung confusion matrix secara manual."""
#     classes = np.unique(y_true)
#     n_classes = len(classes)
#     cm = np.zeros((n_classes, n_classes), dtype=int)
    
#     for i in range(len(y_true)):
#         true_idx = np.where(classes == y_true[i])[0][0]
#         pred_idx = np.where(classes == y_pred[i])[0][0]
#         cm[true_idx][pred_idx] += 1
    
#     return cm

# cm = compute_confusion_matrix(y_test, y_test_pred)
# print(f"\nConfusion Matrix:\n{cm}")

# # Hitung precision, recall, F1-score
# def compute_metrics(cm):
#     """Menghitung metrik evaluasi dari confusion matrix."""
#     tp = cm[1, 1]
#     tn = cm[0, 0]
#     fp = cm[0, 1]
#     fn = cm[1, 0]
    
#     accuracy = (tp + tn) / np.sum(cm)
#     precision = tp / (tp + fp) if (tp + fp) > 0 else 0
#     recall = tp / (tp + fn) if (tp + fn) > 0 else 0
#     f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
#     return {
#         'accuracy': accuracy,
#         'precision': precision,
#         'recall': recall,
#         'f1_score': f1
#     }

# metrics = compute_metrics(cm)
# print(f"\n=== METRIK EVALUASI ===")
# for metric, value in metrics.items():
#     print(f"{metric}: {value:.4f}")

In [19]:
# # Cell 7: Visualisasi Hasil Evaluasi
# fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# # 1. Confusion Matrix
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
#             xticklabels=['Kelas 0', 'Kelas 1'],
#             yticklabels=['Kelas 0', 'Kelas 1'],
#             ax=axes[0, 0])
# axes[0, 0].set_title('Confusion Matrix')
# axes[0, 0].set_xlabel('Prediksi')
# axes[0, 0].set_ylabel('Aktual')

# # 2. Metrik Evaluasi
# metrics_names = list(metrics.keys())
# metrics_values = list(metrics.values())
# axes[0, 1].bar(metrics_names, metrics_values, color=['skyblue', 'lightgreen', 'salmon', 'gold'])
# axes[0, 1].set_title('Metrik Evaluasi Model')
# axes[0, 1].set_ylabel('Nilai')
# axes[0, 1].set_ylim([0, 1])
# for i, v in enumerate(metrics_values):
#     axes[0, 1].text(i, v + 0.02, f'{v:.3f}', ha='center')

# # 3. Feature Importance (probabilitas tertinggi per kelas)
# feature_importance = np.abs(model.feature_probs[0] - model.feature_probs[1])
# top_features_idx = np.argsort(feature_importance)[-10:]  # 10 fitur terpenting
# top_features_names = [f'F{i}' for i in top_features_idx]

# axes[1, 0].barh(top_features_names, feature_importance[top_features_idx], color='teal')
# axes[1, 0].set_title('10 Fitur Terpenting (Berdasarkan Perbedaan Probabilitas)')
# axes[1, 0].set_xlabel('|P(Fitur|Kelas0) - P(Fitur|Kelas1)|')

# # 4. Probabilitas Prediksi untuk beberapa sampel
# probabilities = model.predict_proba(X_test[:10])  # Ambil 10 sampel pertama
# x_pos = np.arange(probabilities.shape[0])
# width = 0.35

# axes[1, 1].bar(x_pos - width/2, probabilities[:, 0], width, label='Kelas 0', alpha=0.7)
# axes[1, 1].bar(x_pos + width/2, probabilities[:, 1], width, label='Kelas 1', alpha=0.7)
# axes[1, 1].set_title('Probabilitas Prediksi (10 Sampel Pertama)')
# axes[1, 1].set_xlabel('Sampel')
# axes[1, 1].set_ylabel('Probabilitas')
# axes[1, 1].set_xticks(x_pos)
# axes[1, 1].legend()

# plt.tight_layout()
# plt.savefig('model_evaluation.png', dpi=100, bbox_inches='tight')
# plt.show()

In [20]:
# # Cell 8: Implementasi GUI untuk Prediksi
# def create_gui(model, feature_names):
#     """
#     Membuat GUI untuk prediksi menggunakan model Naive Bayes.
    
#     Parameters:
#     -----------
#     model : NaiveBayesMultinomial
#         Model yang sudah dilatih
#     feature_names : list
#         Nama-nama fitur
#     """
#     # Fungsi untuk prediksi
#     def predict():
#         try:
#             # Ambil input dari entry
#             input_text = entry.get()
            
#             # Parse input menjadi array
#             input_values = list(map(float, input_text.split(',')))
            
#             if len(input_values) != len(feature_names):
#                 messagebox.showerror("Error", 
#                     f"Harap masukkan tepat {len(feature_names)} nilai, dipisahkan koma")
#                 return
            
#             # Konversi ke numpy array
#             input_array = np.array(input_values).reshape(1, -1)
            
#             # Prediksi
#             prediction = model.predict(input_array)[0]
#             probabilities = model.predict_proba(input_array)[0]
            
#             # Tampilkan hasil
#             result_text = f"Hasil Prediksi: Kelas {prediction}\n\n"
#             result_text += f"Probabilitas:\n"
#             result_text += f"  Kelas 0: {probabilities[0]:.4f}\n"
#             result_text += f"  Kelas 1: {probabilities[1]:.4f}\n\n"
            
#             if prediction == 1:
#                 result_text += "Interpretasi: Data termasuk dalam KELAS POSITIF"
#             else:
#                 result_text += "Interpretasi: Data termasuk dalam KELAS NEGATIF"
            
#             result_label.config(text=result_text)
            
#             # Update progress bar berdasarkan probabilitas
#             progress_bar['value'] = probabilities[1] * 100
            
#         except ValueError:
#             messagebox.showerror("Error", "Harap masukkan angka yang valid, dipisahkan koma")
#         except Exception as e:
#             messagebox.showerror("Error", f"Terjadi kesalahan: {str(e)}")
    
#     # Fungsi untuk generate random input
#     def generate_random():
#         # Generate random binary values (0 atau 1)
#         random_values = np.random.randint(0, 2, len(feature_names))
#         entry.delete(0, tk.END)
#         entry.insert(0, ", ".join(map(str, random_values)))
    
#     # Fungsi untuk reset
#     def reset():
#         entry.delete(0, tk.END)
#         result_label.config(text="Hasil akan muncul di sini")
#         progress_bar['value'] = 50
    
#     # Fungsi untuk melihat info model
#     def show_model_info():
#         info_text = f"=== INFORMASI MODEL ===\n\n"
#         info_text += f"Jumlah fitur: {len(feature_names)}\n"
#         info_text += f"Jumlah kelas: {len(model.classes)}\n"
#         info_text += f"Alpha (smoothing): {model.alpha}\n"
#         info_text += f"Prior probabilities:\n"
#         for i, c in enumerate(model.classes):
#             info_text += f"  Kelas {c}: {model.class_priors[i]:.4f}\n"
        
#         messagebox.showinfo("Informasi Model", info_text)
    
#     # Create main window
#     root = tk.Tk()
#     root.title("Naive Bayes Multinomial Classifier")
#     root.geometry("700x500")
#     root.resizable(False, False)
    
#     # Configure style
#     style = ttk.Style()
#     style.theme_use('clam')
    
#     # Title
#     title_label = tk.Label(root, text="Naive Bayes Multinomial Classifier", 
#                           font=("Arial", 16, "bold"))
#     title_label.pack(pady=10)
    
#     # Input frame
#     input_frame = ttk.Frame(root)
#     input_frame.pack(pady=10, padx=20, fill=tk.X)
    
#     tk.Label(input_frame, text="Masukkan nilai fitur (dipisahkan koma):", 
#             font=("Arial", 10)).pack(anchor=tk.W)
    
#     entry = ttk.Entry(input_frame, width=80)
#     entry.pack(pady=5, fill=tk.X)
    
#     # Button frame
#     button_frame = ttk.Frame(root)
#     button_frame.pack(pady=10)
    
#     ttk.Button(button_frame, text="Prediksi", command=predict, width=15).pack(side=tk.LEFT, padx=5)
#     ttk.Button(button_frame, text="Random Input", command=generate_random, width=15).pack(side=tk.LEFT, padx=5)
#     ttk.Button(button_frame, text="Reset", command=reset, width=15).pack(side=tk.LEFT, padx=5)
#     ttk.Button(button_frame, text="Info Model", command=show_model_info, width=15).pack(side=tk.LEFT, padx=5)
    
#     # Progress bar untuk visualisasi probabilitas
#     progress_frame = ttk.Frame(root)
#     progress_frame.pack(pady=10, padx=20, fill=tk.X)
    
#     tk.Label(progress_frame, text="Tingkat Keyakinan Prediksi Kelas 1:").pack(anchor=tk.W)
#     progress_bar = ttk.Progressbar(progress_frame, length=600, mode='determinate')
#     progress_bar.pack(pady=5)
#     progress_bar['value'] = 50
    
#     # Labels untuk progress bar
#     labels_frame = ttk.Frame(progress_frame)
#     labels_frame.pack(fill=tk.X)
    
#     tk.Label(labels_frame, text="0%", fg="red").pack(side=tk.LEFT)
#     tk.Label(labels_frame, text="50%", fg="orange").pack(side=tk.LEFT, expand=True)
#     tk.Label(labels_frame, text="100%", fg="green").pack(side=tk.RIGHT)
    
#     # Result frame
#     result_frame = ttk.Frame(root, relief=tk.SUNKEN, borderwidth=2)
#     result_frame.pack(pady=10, padx=20, fill=tk.BOTH, expand=True)
    
#     result_label = tk.Label(result_frame, text="Hasil akan muncul di sini", 
#                            font=("Arial", 11), justify=tk.LEFT, anchor=tk.NW, wraplength=650)
#     result_label.pack(pady=10, padx=10, fill=tk.BOTH, expand=True)
    
#     # Status bar
#     status_bar = tk.Label(root, text=f"Siap. Model dengan {len(feature_names)} fitur", 
#                          bd=1, relief=tk.SUNKEN, anchor=tk.W)
#     status_bar.pack(side=tk.BOTTOM, fill=tk.X)
    
#     return root

# # Dapatkan nama fitur (kolom tanpa label)
# feature_names = df.columns[:-1].tolist()

# # Jalankan GUI
# gui_root = create_gui(model, feature_names)
# gui_root.mainloop()